# Section 3: Techniques to Improve Gen AI Model Output
## GCP Generative AI Leader Certification

This notebook covers hands-on exercises for improving generative AI output:
- Prompt engineering: zero-shot, one-shot, few-shot
- Role prompting and system instructions
- Chain-of-thought (CoT) prompting
- ReAct pattern demonstration
- Temperature and sampling parameter comparison
- Grounding with Google Search
- RAG implementation patterns
- Fine-tuning concepts

**Prerequisites**: A GCP project with Vertex AI API enabled and billing configured.

---
## 1. Environment Setup

In [ ]:
!pip install -q google-cloud-aiplatform>=1.60.0 google-generativeai>=0.7.0

In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab.")
except ImportError:
    print("Not in Colab. Using default credentials.")

PROJECT_ID = "your-project-id"  # <-- Replace
LOCATION = "us-central1"

import vertexai
vertexai.init(project=PROJECT_ID, location=LOCATION)

from vertexai.generative_models import GenerativeModel, GenerationConfig
print(f"Ready. Project: {PROJECT_ID}")

---
## 2. Zero-Shot vs Few-Shot Prompting

Compare how adding examples affects output quality and consistency.

In [ ]:
# Zero-shot: No examples
zero_shot = """Extract the product name, price, and sentiment from this review.

Review: "I bought the Sony WH-1000XM5 headphones for $349 and they are absolutely incredible. 
The noise cancellation is the best I've ever experienced."

Extraction:"""

# Few-shot: With examples
few_shot = """Extract the product name, price, and sentiment from each review.

Review: "The Apple AirPods Pro 2 at $249 are decent but the fit could be better."
Extraction: {"product": "Apple AirPods Pro 2", "price": "$249", "sentiment": "MIXED"}

Review: "Terrible experience with the $99 Skullcandy earbuds. They broke in a week."
Extraction: {"product": "Skullcandy earbuds", "price": "$99", "sentiment": "NEGATIVE"}

Review: "I bought the Sony WH-1000XM5 headphones for $349 and they are absolutely incredible. 
The noise cancellation is the best I've ever experienced."
Extraction:"""

try:
    model = GenerativeModel("gemini-1.5-pro")
    
    print("=== Zero-Shot Result ===")
    resp_zero = model.generate_content(zero_shot)
    print(resp_zero.text)
    
    print("\n=== Few-Shot Result ===")
    resp_few = model.generate_content(few_shot)
    print(resp_few.text)

except Exception as e:
    print(f"API note: {e}")
    print("\n--- Mock Output ---")
    print("=== Zero-Shot Result ===")
    print("Product: Sony WH-1000XM5 headphones")
    print("Price: $349")
    print("Sentiment: Positive")
    print("(Note: Output format is inconsistent - free-form text)")
    print("\n=== Few-Shot Result ===")
    print('{"product": "Sony WH-1000XM5", "price": "$349", "sentiment": "POSITIVE"}')
    print("(Note: Output matches the JSON format from examples - consistent!)")
    print("\n** Key insight: Few-shot examples enforce consistent output format. **")

---
## 3. Role Prompting (System Instructions)

System instructions set the model's persona, expertise, and behavior guidelines.
They persist across all turns in a conversation.

In [ ]:
# Compare responses with different roles
question = "Should our company invest in generative AI?"

roles = {
    "CFO Advisor": "You are a Chief Financial Officer advisor. Focus on ROI, costs, "
                   "budget allocation, and financial risk. Be quantitative and conservative.",
    "CTO Advisor": "You are a Chief Technology Officer advisor. Focus on technical "
                   "architecture, scalability, team skills, and innovation potential.",
    "CISO Advisor": "You are a Chief Information Security Officer advisor. Focus on "
                    "security risks, data privacy, compliance, and threat mitigation."
}

try:
    for role_name, instruction in roles.items():
        model = GenerativeModel("gemini-1.5-pro", system_instruction=instruction)
        resp = model.generate_content(question)
        print(f"\n{'='*50}")
        print(f"Role: {role_name}")
        print(f"{'='*50}")
        print(resp.text[:300] + "...")

except Exception as e:
    print(f"API note: {e}")
    print("\n--- Mock Output ---")
    print("\nRole: CFO Advisor")
    print("Start with low-risk pilot ($50-100K). Expect 6-12 month payback.")
    print("Focus on measurable productivity gains. Gemini for Workspace at")
    print("$30/user/month is the lowest-risk entry point.")
    print("\nRole: CTO Advisor")
    print("Adopt Vertex AI as your AI platform. Start with prompt engineering,")
    print("then add RAG for knowledge grounding. Plan for fine-tuning in Phase 2.")
    print("Build an AI platform team of 3-5 engineers.")
    print("\nRole: CISO Advisor")
    print("Implement Google's SAIF framework. Ensure VPC-SC, CMEK, DLP.")
    print("Verify data processing agreements. Assess prompt injection risks.")
    print("Start with Vertex AI (enterprise controls) not consumer APIs.")
    print("\n** Key insight: Same question, vastly different answers based on role. **")

---
## 4. Chain-of-Thought Prompting

CoT dramatically improves performance on reasoning and math tasks
by forcing the model to show its work.

In [ ]:
# Problem that benefits from step-by-step reasoning
problem = """A company currently spends $500,000/year on customer support with 20 agents.
They plan to deploy an AI chatbot that can handle 60% of Tier 1 tickets.
The AI costs $8,000/month. They can reduce to 12 agents.
Each agent costs $25,000/year.
What are the annual savings after deploying AI?"""

# Without CoT
direct_prompt = problem + "\n\nAnswer:"

# With CoT
cot_prompt = problem + "\n\nLet's solve this step by step:"

try:
    model = GenerativeModel("gemini-1.5-pro")
    
    print("=== Without Chain-of-Thought ===")
    resp_direct = model.generate_content(direct_prompt)
    print(resp_direct.text[:300])
    
    print("\n=== With Chain-of-Thought ===")
    resp_cot = model.generate_content(cot_prompt)
    print(resp_cot.text[:500])

except Exception as e:
    print(f"API note: {e}")
    print("\n--- Mock Output ---")
    print("=== Without Chain-of-Thought ===")
    print("The annual savings would be approximately $104,000.")
    print("(May give wrong answer without showing work)")
    print("\n=== With Chain-of-Thought ===")
    print("Step 1: Current cost = $500,000/year (20 agents)")
    print("Step 2: New agent cost = 12 agents x $25,000 = $300,000/year")
    print("Step 3: AI cost = $8,000/month x 12 = $96,000/year")
    print("Step 4: New total = $300,000 + $96,000 = $396,000/year")
    print("Step 5: Savings = $500,000 - $396,000 = $104,000/year")
    print("\nThe company saves $104,000/year (20.8% reduction).")
    print("\n** CoT makes reasoning visible and verifiable. **")

---
## 5. Temperature and Sampling Comparison

See how temperature and top-p affect output diversity and quality.

In [ ]:
# Compare temperature settings on the same prompt
creative_prompt = "Write a one-sentence tagline for an AI-powered coffee shop called 'Neural Brew'."

temperatures = [0.0, 0.3, 0.7, 1.0, 1.5]

print("=== Temperature Comparison ===")
print(f"Prompt: {creative_prompt}\n")

try:
    model = GenerativeModel("gemini-1.5-pro")
    for temp in temperatures:
        config = GenerationConfig(temperature=temp, max_output_tokens=100)
        # Generate 2 responses at each temperature to show variation
        responses = []
        for _ in range(2):
            resp = model.generate_content(creative_prompt, generation_config=config)
            responses.append(resp.text.strip())
        print(f"Temp {temp}:")
        for r in responses:
            print(f"  -> {r}")
        same = responses[0] == responses[1]
        print(f"  (Identical: {same})\n")

except Exception as e:
    print(f"API note: {e}")
    print("\n--- Mock Output ---")
    print("Temp 0.0:")
    print("  -> Where every cup is brewed by intelligence.")
    print("  -> Where every cup is brewed by intelligence.")
    print("  (Identical: True - deterministic!)\n")
    print("Temp 0.3:")
    print("  -> Where every cup is brewed by intelligence.")
    print("  -> Where every sip sparks a new idea.")
    print("  (Identical: False - slight variation)\n")
    print("Temp 0.7:")
    print("  -> Coffee that learns how you like it.")
    print("  -> Your morning boost, optimized by AI.")
    print("  (Identical: False - good variety)\n")
    print("Temp 1.0:")
    print("  -> Sip the future, one algorithm at a time.")
    print("  -> We don't predict the future, we brew it.")
    print("  (Identical: False - creative variety)\n")
    print("Temp 1.5:")
    print("  -> Caffeinated synapses firing in harmony with your soul.")
    print("  -> Deep roast, deeper learning, deepest satisfaction.")
    print("  (Identical: False - very creative, may lose coherence)")

In [ ]:
# Compare top-p settings
factual_prompt = "What is Google Cloud's Vertex AI?"

top_p_values = [0.1, 0.5, 0.95]

print("=== Top-P Comparison ===")
print(f"Prompt: {factual_prompt}\n")

try:
    model = GenerativeModel("gemini-1.5-pro")
    for tp in top_p_values:
        config = GenerationConfig(temperature=0.7, top_p=tp, max_output_tokens=150)
        resp = model.generate_content(factual_prompt, generation_config=config)
        print(f"Top-P {tp}: {resp.text.strip()[:200]}")
        print()

except Exception as e:
    print(f"API note: {e}")
    print("\n--- Mock Output ---")
    print("Top-P 0.1 (focused): Vertex AI is Google Cloud's unified ML platform.")
    print("  (Very focused, picks most likely tokens)")
    print("")
    print("Top-P 0.5 (balanced): Vertex AI is Google Cloud's comprehensive AI")
    print("  development platform for building and deploying ML models.")
    print("  (Moderate diversity in word choice)")
    print("")
    print("Top-P 0.95 (diverse): Vertex AI represents Google Cloud's all-in-one")
    print("  artificial intelligence ecosystem designed to streamline the entire")
    print("  machine learning lifecycle from experimentation to production.")
    print("  (More diverse vocabulary, longer descriptions)")

### Sampling Parameter Cheat Sheet

| Scenario | Temperature | Top-P | Top-K |
|----------|------------|-------|-------|
| Factual Q&A | 0.0-0.2 | 0.8 | 20 |
| Code generation | 0.2-0.4 | 0.9 | 40 |
| Conversational | 0.5-0.7 | 0.9 | 40 |
| Creative writing | 0.8-1.0 | 0.95 | 50 |
| Brainstorming | 1.0-1.5 | 0.99 | 100 |

---
## 6. ReAct Pattern (Reasoning + Acting)

The ReAct pattern alternates between reasoning and tool use.
This is the foundation for AI agents.

In [ ]:
# Simulate a ReAct agent with manual tool execution

# Simulated tools
def search_knowledge_base(query):
    """Simulated knowledge base search"""
    kb = {
        "pricing": "Vertex AI pricing: Gemini Pro $0.00125/1K chars input, $0.00375/1K chars output. Flash: 10x cheaper.",
        "limits": "Gemini 1.5 Pro: 2M token context, 8K output. Flash: 1M context, 8K output.",
        "regions": "Vertex AI Gemini available in: us-central1, europe-west4, asia-northeast1."
    }
    for key, value in kb.items():
        if key in query.lower():
            return value
    return "No relevant information found."

def calculate(expression):
    """Simulated calculator"""
    try:
        return str(eval(expression))
    except:
        return "Calculation error."

# Manual ReAct simulation
print("=== ReAct Pattern Simulation ===")
print("User: How much would it cost to process 1 million characters with Gemini Pro?\n")

print("Thought 1: I need to find the pricing for Gemini Pro.")
print("Action 1: search_knowledge_base('pricing')")
result1 = search_knowledge_base("pricing")
print(f"Observation 1: {result1}\n")

print("Thought 2: Gemini Pro costs $0.00125/1K chars input. For 1M chars, I need to calculate.")
print("Action 2: calculate('1000000 / 1000 * 0.00125')")
result2 = calculate("1000000 / 1000 * 0.00125")
print(f"Observation 2: {result2}\n")

print("Thought 3: The input cost is $1.25. I should also calculate output cost.")
print("Action 3: calculate('1000000 / 1000 * 0.00375')")
result3 = calculate("1000000 / 1000 * 0.00375")
print(f"Observation 3: {result3}\n")

print("Final Answer: Processing 1 million characters with Gemini Pro costs:")
print(f"  - Input: ${result2}")
print(f"  - Output: ${result3}")
print("  Gemini Flash would be approximately 10x cheaper.")
print("\n** ReAct enables agents to break down problems and use tools. **")

---
## 7. Grounding vs No Grounding

Compare the factual accuracy of grounded vs ungrounded responses.

In [ ]:
# Demonstrate the value of grounding
question = "What is Google Cloud's latest TPU offering and its performance specs?"

try:
    model = GenerativeModel("gemini-1.5-pro")
    
    # Without grounding (may hallucinate or be outdated)
    print("=== Without Grounding ===")
    resp_ungrounded = model.generate_content(question)
    print(resp_ungrounded.text[:300])
    print("\n(Warning: This may contain outdated or hallucinated information)")
    
    # With Google Search grounding
    print("\n=== With Google Search Grounding ===")
    from vertexai.generative_models import Tool, grounding
    search_tool = Tool.from_google_search_retrieval(
        grounding.GoogleSearchRetrieval()
    )
    resp_grounded = model.generate_content(question, tools=[search_tool])
    print(resp_grounded.text[:300])
    print("\n(Grounded in current web search results - more reliable)")

except Exception as e:
    print(f"API note: {e}")
    print("\n--- Mock Output ---")
    print("=== Without Grounding ===")
    print("Google Cloud's latest TPU is the TPU v4, which offers...")
    print("(OUTDATED - the model's training data may not include v5e/v5p)\n")
    print("=== With Google Search Grounding ===")
    print("Google Cloud's latest TPU offerings include TPU v5e (cost-efficient)")
    print("and TPU v5p (maximum performance). The v5p offers 2x the FLOPS")
    print("of v4 and 3x the HBM capacity...")
    print("Source: cloud.google.com/tpu")
    print("(CURRENT - grounded in live web search results)")
    print("\n** Grounding prevents hallucinations and knowledge staleness. **")

---
## 8. Technique Selection Guide

When to use which technique - the key decision framework for the exam.

In [ ]:
# Decision framework as a reference table
techniques = [
    ["Prompt Engineering", "Low", "Minutes", "Free", "Format, tone, basic behavior"],
    ["System Instructions", "Low", "Minutes", "Free", "Consistent persona/role"],
    ["Few-Shot Examples", "Low", "Minutes", "Tokens only", "Output format, classification"],
    ["Chain-of-Thought", "Low", "Minutes", "Tokens only", "Reasoning, math, logic"],
    ["Google Search Ground.", "Low", "Minutes", "Per-query", "Current events, public facts"],
    ["RAG (Custom Data)", "Medium", "Days", "Index + query", "Enterprise docs, private data"],
    ["Fine-Tuning (SFT)", "High", "Weeks", "Training + $", "Domain language, style, behavior"],
    ["RLHF", "Very High", "Months", "Expensive", "Alignment, safety, preferences"],
    ["Distillation", "High", "Weeks", "Training + $", "Smaller model with similar quality"],
]

print("=== Technique Selection Guide ===")
print(f"{'Technique':<22} {'Effort':<10} {'Time':<10} {'Cost':<14} {'Best For'}")
print("-" * 90)
for t in techniques:
    print(f"{t[0]:<22} {t[1]:<10} {t[2]:<10} {t[3]:<14} {t[4]}")

print("\n** Start at the top (cheapest/fastest) and move down only if needed. **")
print("The exam expects you to recommend the SIMPLEST technique that solves the problem.")

---
## Summary

In this notebook, you practiced:

1. **Zero-Shot vs Few-Shot** - Few-shot enforces consistent output format
2. **Role Prompting** - System instructions shape model behavior dramatically
3. **Chain-of-Thought** - Step-by-step reasoning improves accuracy on complex tasks
4. **ReAct Pattern** - Reasoning + acting enables tool-using agents
5. **Temperature** - 0 = deterministic, higher = more creative
6. **Top-P / Top-K** - Control token selection diversity
7. **Grounding** - Connects model to current, factual information
8. **Technique Selection** - Start simple (prompt engineering), escalate only if needed

**Key exam takeaways:**
- Prompt engineering is the first technique to try (fast, free)
- RAG for factual accuracy with custom data; fine-tuning for behavior changes
- Temperature 0 for factual tasks, 0.7-1.0 for creative tasks
- CoT for reasoning; ReAct for agent-style tool use
- Always recommend the simplest sufficient technique

**Next**: [Section 4 - Business Strategies for Gen AI](04-business-strategies.ipynb)